# Time is the Hardest Dimension

Time dimensions are special: they're naturally ordered, and they need an aggregation granularity (daily, weekly, monthly). This makes them the foundation for a family of transforms that are simple to express but non-trivial to implement in SQL.

This notebook demonstrates SLayer's time-related features with working code.

See also: [Formulas](../../concepts/formulas.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Time Dimensions and Granularity

A time dimension truncates a date/timestamp column to a given granularity. SLayer supports `day`, `week`, `month`, `quarter`, and `year`.

Let's start with monthly revenue:

In [2]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "measures": ["sum(order_total)"],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14}")
print("-" * 28)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} ${rev:>13,.2f}")

Month               Revenue
----------------------------
2023-11      $    16,062.47
2023-12      $    24,029.92
2024-01      $    30,308.25
2024-02      $    35,652.23
2024-03      $    43,155.56
2024-04      $    46,032.07
2024-05      $    73,399.17
2024-06      $    90,649.17
2024-07      $   100,621.34
2024-08      $   110,755.70
2024-09      $    93,833.16
2024-10      $   103,870.50


## Month-over-Month Change

The `change()` transform computes the difference from the previous period. It needs no extra arguments — the time dimension and granularity are already known from the query. Internally it uses a self-join CTE (not LAG), so it handles time gaps correctly: the previous period is matched on the calendar (e.g., the month before), not on the previous *row* in the result set. The very first period in the window still has no prior period to compare against, so its change is NULL (the first row below shows `-`); subsequent periods are gap-safe.

`change_pct()` gives the percentage change:

In [3]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "measures": [
            "sum(order_total)",
            {"formula": "change(sum(order_total))", "name": "mom_change"},
            {"formula": "change_pct(sum(order_total))", "name": "mom_pct"},
        ],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'MoM Change':>14} {'MoM %':>8}")
print("-" * 52)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>13,.2f} {chg_str:>14} {pct_str:>8}")

Month               Revenue     MoM Change    MoM %
----------------------------------------------------
2023-11      $    16,062.47              -        -
2023-12      $    24,029.92        $+7,967   +49.6%
2024-01      $    30,308.25        $+6,278   +26.1%
2024-02      $    35,652.23        $+5,344   +17.6%
2024-03      $    43,155.56        $+7,503   +21.0%
2024-04      $    46,032.07        $+2,877    +6.7%
2024-05      $    73,399.17       $+27,367   +59.5%
2024-06      $    90,649.17       $+17,250   +23.5%
2024-07      $   100,621.34        $+9,972   +11.0%
2024-08      $   110,755.70       $+10,134   +10.1%
2024-09      $    93,833.16       $-16,923   -15.3%
2024-10      $   103,870.50       $+10,037   +10.7%


## Year-over-Year with time_shift

`time_shift(measure, offset, granularity)` shifts the time dimension by the given offset before applying the query's granularity. This is implemented as a calendar-based self-join CTE — semantically simple, but the SQL is non-trivial.

For example, `time_shift(order_total_sum, -1, 'year')` gives "same month, previous year":

In [4]:
import datetime as dt

# Use last full calendar year so the example stays in the demo's data window
# (the loader shifts dates so MAX(ordered_at) == today, with ~4 years of history).
last_year = dt.date.today().year - 1
date_range = [f"{last_year}-01-01", f"{last_year}-12-31"]

result = engine.execute_sync(
    query={
        "source_model": "orders",
        "time_dimensions": [
            {
                "dimension": "ordered_at",
                "granularity": "month",
                "date_range": date_range,
            }
        ],
        "measures": [
            "sum(order_total)",
            {"formula": "time_shift(sum(order_total), -1, 'year')", "name": "prev_year"},
        ],
        "order": [{"column": "ordered_at", "direction": "asc"}],
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Prev Year':>14}")
print("-" * 42)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    prev = row["orders.prev_year"]
    prev_str = f"${prev:>13,.2f}" if prev is not None else f"{'n/a':>14}"
    print(f"{month:<12} ${rev:>13,.2f} {prev_str}")

Month               Revenue      Prev Year
------------------------------------------
2025-01      $   174,373.56 $    30,308.25
2025-02      $   163,543.36 $    35,652.23
2025-03      $   184,439.93 $    43,155.56
2025-04      $   183,467.82 $    46,032.07
2025-05      $   192,061.70 $    73,399.17
2025-06      $   177,772.33 $    90,649.17
2025-07      $   236,307.35 $   100,621.34
2025-08      $   231,640.58 $   110,755.70
2025-09      $   195,181.77 $    93,833.16
2025-10      $   221,536.41 $   103,870.50
2025-11      $   248,542.85 $   121,048.89
2025-12      $   333,418.92 $   162,339.92


### time_shift vs lag

`time_shift` always uses a calendar-based self-join, with or without a granularity argument. Without one, `time_shift(x, -1)` shifts by one bucket of the query's own time granularity; with one (like `'year'` above), the shift can be any calendar amount — not just a multiple of the query granularity. Either way it's gap-safe: the previous value is matched on the actual calendar date, so a missing intermediate month produces NULL rather than the wrong value. The join also matches on all non-time dimensions, so grouped series (e.g. per-store) reset cleanly per group.

`lag(x, 1)` / `lead(x, 1)` use SQL's `LAG()`/`LEAD()` window functions instead — more efficient, but they blindly walk rows: if the result has a gap (e.g., a missing month), the previous row is the wrong period. Both approaches return NULL at the start of the window when there's no prior period in the data to match.

## The last() Transform

`last(measure)` returns the most recent time period's value, broadcast to every row. This **collapses the time dimension** — useful for comparing each period to the latest one, or for filtering.

For example, revenue by store alongside each store's most recent month's revenue:

In [5]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "quarter"}],
        "measures": [
            "sum(order_total)",
            {"formula": "last(sum(order_total))", "name": "latest_quarter"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"column": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'Latest Qtr':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    latest = row["orders.latest_quarter"]
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} ${latest:>13,.2f}")

Quarter      Store                       Revenue     Latest Qtr
--------------------------------------------------------------
2026-10      San Francisco        $    80,540.76 $    80,540.76
2026-10      New Orleans          $    44,125.00 $    44,125.00
2026-10      Brooklyn             $   100,513.60 $   100,513.60
2026-10      Philadelphia         $    60,541.17 $    60,541.17
2026-10      Chicago              $    86,955.69 $    86,955.69
2026-07      Brooklyn             $   316,092.65 $   100,513.60
2026-07      Chicago              $   272,179.96 $    86,955.69
2026-07      New Orleans          $    92,060.61 $    44,125.00
2026-07      Philadelphia         $   191,060.10 $    60,541.17
2026-07      San Francisco        $   244,313.76 $    80,540.76


## Composing Transforms in Filters

Transforms can be composed and used in filters. For example, `last(change(order_total_sum)) < 0` keeps only rows where the most recent period's change was negative — i.e., revenue declined in the latest period.

This is a powerful pattern: "show me all stores, but only those whose most recent quarter-over-quarter revenue went down":

In [6]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "quarter"}],
        "measures": [
            "sum(order_total)",
            {"formula": "change(sum(order_total))", "name": "qoq_change"},
        ],
        "dimensions": ["stores.name"],
        "filters": ["last(change(sum(order_total))) < 0"],
        "order": [{"column": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print("Stores where latest QoQ revenue declined:")
print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'QoQ Change':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    chg = row["orders.qoq_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} {chg_str:>14}")

Stores where latest QoQ revenue declined:
Quarter      Store                       Revenue     QoQ Change
--------------------------------------------------------------
2026-10      Brooklyn             $   100,513.60      $-215,579
2026-10      New Orleans          $    44,125.00       $-47,936
2026-10      Philadelphia         $    60,541.17      $-130,519
2026-10      San Francisco        $    80,540.76      $-163,773
2026-10      Chicago              $    86,955.69      $-185,224
2026-07      Philadelphia         $   191,060.10       $-27,029
2026-07      San Francisco        $   244,313.76       $-14,055
2026-07      New Orleans          $    92,060.61       $+53,927
2026-07      Brooklyn             $   316,092.65       $-46,079
2026-07      Chicago              $   272,179.96       $-24,586


## Summary

Each of these concepts is simple and natural at the semantic level, but requires non-trivial SQL (CTEs, self-joins, window functions) to implement. SLayer offloads that effort.

| Transform | What it does | SQL implementation |
|-----------|-------------|-------------------|
| `change(x)` | Difference from previous period | Self-join CTE (gap-safe; first period is NULL) |
| `change_pct(x)` | Percentage change from previous period | Same as change, divided |
| `time_shift(x, -1, 'year')` | Same measure, shifted in time | Calendar-based self-join CTE |
| `lag(x)` / `lead(x)` | Previous/next row's value | SQL LAG/LEAD window (gap-blind) |
| `last(x)` | Most recent period's value, broadcast | Sub-query with FIRST_VALUE |
| `cumsum(x)` | Running total over time | SUM window function |

All transforms compose: `cumsum(change(x))`, `last(change(x))` in filters, etc.

See the [Formulas docs](../../concepts/formulas.md) for the full reference.